<a href="https://colab.research.google.com/github/AArashinAA/Crypto/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:

!pip install chromaDB

In [14]:
import os
import docx
# import pandas as pd
# import numpy as np
import PyPDF2
import chromadb
from chromadb.utils import embedding_functions

In [6]:
def read_text_file(filePath):
    with open(filePath, 'r', encoding='utf-8') as f:
        return f.read()

In [9]:
def read_pdf_file(filePath):
    with open(filePath, 'rb') as f:
        pdfReader = PyPDF2.PdfReader(f)
        text = ''
        for page in pdfReader.pages:
            text += page.extract_text() + '\n'
        return text

In [10]:
def read_docx_file(file_path):
    doc = docx.Document(file_path)
    text = '\n'.join([paragraph.text for paragraph in doc.paragraphs])
    return text

In [11]:
def read_document(filePath):
    if filePath.endswith('.txt'):
        return read_text_file(filePath)
    elif filePath.endswith('.pdf'):
        return read_pdf_file(filePath)
    elif filePath.endswith('.docx'):
        return read_docx_file(filePath)

In [12]:
def split_text(text, chunk_size=500, chunk_overlap=100):
    # splitting text into overlapping chunks
    text = text.replace('\n', ' ')
    start = 0
    length = len(text)
    chunks = []
    while start < length:
        end = min(start + chunk_size, length)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - chunk_overlap

    return chunks

In [30]:
file_path = './drylab.pdf'
text = read_document(file_path)
print(text[:500])

Drylab Newsfor investors & friends · May 2 017
Welcome to our first newsletter of 2017! It's
been a while since the last one, and a lot has
happened. W e promise to k eep them coming
every two months hereafter , and permit
ourselv es to mak e this one r ather long. The
big news is the beginnings of our launch in
the American mark et, but there are also
interesting updates on sales, de velopment,
mentors and ( of course ) the in vestment
round that closed in January .
New capital: The in vestment


In [32]:
chunks = split_text(text)

In [34]:
print(len(chunks))

16


In [17]:
client = chromadb.PersistentClient(path="./chorama_db")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2")
collection = client.get_or_create_collection(
    name="my_collection",
    embedding_function=sentence_transformer_ef)